# 🤖 03. Machine Learning Model Training & Comparison
### TY BSc Data Science Final-Year Project
**Objective:** Train four tabular algorithms (Logistic Regression, Decision Tree, Random Forest, XGBoost), execute 5-fold Stratified Cross-Validation, and address class imbalance.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import joblib

sys.path.append(os.path.abspath(".."))
from src.preprocessing import prepare_training_data
from src.train_model import build_candidate_models
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Imports ready.")

## 1. Load Preprocessed Splits

In [2]:
raw_data_path = os.path.join("..", "data", "raw", "customer_churn.csv")
X_train, X_test, y_train, y_test, preprocessor, feature_names, _ = prepare_training_data(raw_data_path)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"Class distribution in train - Retained: {(y_train == 0).sum()}, Churned: {(y_train == 1).sum()}")

## 2. Define Algorithms & Compute 5-Fold Cross-Validation

In [3]:
models = build_candidate_models(y_train)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for name, model in models.items():
    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    roc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        "Model": name,
        "CV F1 (Mean)": round(float(np.mean(f1_scores)), 4),
        "CV F1 (Std)": round(float(np.std(f1_scores)), 4),
        "CV ROC-AUC (Mean)": round(float(np.mean(roc_scores)), 4),
        "CV ROC-AUC (Std)": round(float(np.std(roc_scores)), 4)
    })

cv_df = pd.DataFrame(cv_results)
cv_df

## 3. Fit Models on Training Set

In [4]:
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"Trained: {name}")

## 4. Persist Artifacts

In [5]:
models_dir = os.path.join("..", "models")
joblib.dump(trained_models, os.path.join(models_dir, "all_models.pkl"))
print("Saved all models to models/all_models.pkl")